# 🤖 03 — Pemodelan (Model Training)
> Notebook ini melatih tiga model: **SVR**, **Random Forest**, dan **Linear Regression**.
> Model terbaik disimpan ke `models/`.

## 📦 Import Library

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('..')

import pandas as pd

from src.data_loader  import load_config, load_processed_train
from src.train        import get_models, split_data, train_all_models, save_model
from src.evaluate     import compute_metrics

print('✅ Library berhasil diimpor.')

## 📂 Muat Data Terproses

In [ ]:
cfg      = load_config('../config.yaml')
df       = load_processed_train(f"../{cfg['data']['train_processed']}")
df.head(3)

## ✂️ 5. Pembagian Train / Validasi

In [ ]:
target = cfg['data']['target_column']
X = df.drop(columns=[target])
y = df[target]

X_train, X_valid, y_train, y_valid = split_data(
    X, y,
    test_size    = cfg['preprocessing']['test_size'],
    random_state = cfg['preprocessing']['random_state'],
)
print(f'Jumlah fitur : {X.shape[1]}')

## 🤖 6. Pelatihan Model
Tiga algoritma dibandingkan:
- **SVR** (Support Vector Regressor) — kernel RBF
- **Random Forest** — 200 estimator
- **Linear Regression** — baseline

In [ ]:
models = get_models(cfg['models'])
print('Model yang akan dilatih:')
for name in models:
    print(f'  • {name}')

In [ ]:
trained_models, preds = train_all_models(models, X_train, y_train, X_valid)

## 📊 6.1 Evaluasi Awal (Validasi)

In [ ]:
results = []
for name, y_pred in preds.items():
    results.append(compute_metrics(y_valid, y_pred, name))

results_df = pd.DataFrame(results).set_index('Model')
results_df

## 💾 6.2 Simpan Model Terbaik (Random Forest)

In [ ]:
model_path = cfg['output']['model_path']
save_model(trained_models['Random Forest'], f'../{model_path}')
print(f'Model disimpan ke: {model_path}')

## ✅ Ringkasan

In [ ]:
best = results_df['R²'].idxmax()
print(f'🏆 Model terbaik : {best}')
print(f'   R²            : {results_df.loc[best, "R²"]:.4f}')
print(f'   MAPE          : {results_df.loc[best, "MAPE (%)"]:.2f}%')
print(f'   MAE           : ${results_df.loc[best, "MAE ($)"]:,.0f}')